In [2]:
import os
import geopandas as gpd
import pandas as pd
import numpy as np
from glob import glob
import rasterio as rio
from rasterio.mask import mask
from rasterio.io import MemoryFile
import pickle
import matplotlib.pyplot as plt
from concurrent.futures import ThreadPoolExecutor
from threading import Lock
from src.mslandcover.config import MSTM_PROJ4
from src.mslandcover.utils import raise_if_not_exists
import cv2

In [60]:
# load the shapefilse with boundaries of the regions
shapefiles = glob('data/MS_NAIP_2023/*/*.shp')
gdfs = []
for shapefile in shapefiles:
    gdf = gpd.read_file(shapefile).to_crs(MSTM_PROJ4) # convert to the same projection
    raster_path = glob(os.path.join(os.path.dirname(shapefile), '*_1m.tif'))[0]
    gdf['raster_path'] = raster_path
    gdfs.append(gdf)

raster_boundaries_gdf = gpd.GeoDataFrame(pd.concat(gdfs))[['raster_path', 'geometry']] # only keep the relevant columns
raster_boundaries_gdf = raster_boundaries_gdf.dissolve(by='raster_path').reset_index() # dissolve the geometries to get the boundaries of the raster
raster_boundaries_gdf.to_file('data/regions_boundaries.gpkg', driver='GPKG')

In [40]:
# load the samples parquet
samples = gpd.read_parquet('./data/sampling/samples.par')

# spatial join with the raster boundaries
raster_boundaries_gdf['raster_geometry'] = raster_boundaries_gdf['geometry'] # make a copy of the geometry
samples = gpd.sjoin(samples, raster_boundaries_gdf, predicate='intersects', how='left')
samples['intersection_area'] = samples.apply(lambda x: x['geometry'].intersection(x['raster_geometry']).area, axis=1)

In [ ]:
def extract_mask(sample, raster_dataset):
    
    out_image, out_transform = mask(raster_dataset, [sample['geometry']], crop=True, all_touched=True)
    out_meta = raster_dataset.meta.copy()

    if out_image.shape[1] > 256 or out_image.shape[2] > 256:
        # crop the image to 256x256
        out_image = out_image[:, :256, :256]

    filename = str(sample.name)
    if out_image.shape != (3, 256, 256):\
        return
    
    nd_values = np.array([raster_dataset.nodata] * 3)
    pixels_with_nd = np.equal(out_image.transpose(1, 2, 0).reshape(-1, 3), nd_values).all(axis=1)
    if pixels_with_nd.any():
        
        if pixels_with_nd.sum() > 0.05 * len(pixels_with_nd):
            return
    
    out_meta.update({
        "driver": "GTiff",
        "height": out_image.shape[1],
        "width": out_image.shape[2],
        "transform": out_transform,
    })
    
    out_path = os.path.join('data', 'splits', sample['split'], 'input', filename + '.tif')
    with rio.open(out_path, 'w', **out_meta) as dst:
        dst.write(out_image)
    
    if sample['split'] == 'pretrain' or sample['split'] == 'pretrain_val': # save HSV image for pretraining
        hsv_image = cv2.cvtColor(out_image.transpose(1, 2, 0), cv2.COLOR_RGB2HSV).transpose(2, 0, 1)
        
        target_path = os.path.join('data', 'splits', sample['split'], 'target', filename + '.tif')
        with rio.open(target_path, 'w', **out_meta) as dst:
            dst.write(hsv_image)

def extract_raster(samples_group):
    
    raster_path = samples_group[0]
    with MemoryFile(open(raster_path, 'rb')) as memfile:
        with rio.open(memfile) as raster_dataset:
            samples_group[1].apply(lambda x: extract_mask(x, raster_dataset), axis=1)

# sub_samples = samples[samples['split'].isin(['train', 'test', 'val'])]
n_threads = 4
with ThreadPoolExecutor(max_workers=n_threads) as executor:
    list(executor.map(extract_raster, list(samples.groupby('raster_path'))))

In [ ]:
# check which images did not get sampled - these most likely have pathces in 
# multiple rasters which need to be handled separately
sampled_files = [int(os.path.basename(file).replace('.tif', '')) for file in glob('data/splits/*/*/*.tif')]

Index([1590368,  631402,  631402,  451144, 1751567, 1751567, 1966146, 1633483,
       1633483, 1633483,
       ...
       1992753, 1897841, 2407332, 2267476, 1524028, 1524028, 1524028,  624760,
       2033673, 2068866],
      dtype='int64', length=1373)
0


,geometry,hist_vector,hist_vector_scaled,hist_vector_pca,cluster,split,index_right,raster_path,raster_geometry,intersection_area


In [51]:
def combine_images(gdf):
    print(gdf)


buffered_raster_footprints = raster_boundaries_gdf.copy()
buffered_raster_footprints['geometry'] = buffered_raster_footprints['geometry'].buffer(100)

remaining_samples = gpd.sjoin(remaining_samples.drop(['index_right'], axis=1), buffered_raster_footprints, predicate='intersects', how='left')
display(remaining_samples)

,geometry,hist_vector,hist_vector_scaled,hist_vector_pca,cluster,split,raster_path_left,raster_geometry_left,intersection_area,index_right,raster_path_right,raster_geometry_right
631402,"POLYGON ((395614.025 1480093.786, 395870.025 1...","[0.20833333333333334, 0.0, 0.0, 0.0, 0.0, 0.0,...","[0.20833333333333334, 0.0, 0.0, -0.33333333333...","[-1.461855507383653, 2.5955461354892018]",0,train,data/MS_NAIP_2023\ortho_1-1_hc_s_ms011_2023_1\...,"POLYGON ((360356.027 1411250.111, 360235.899 1...",65536.0,5,data/MS_NAIP_2023\ortho_1-1_hc_s_ms011_2023_1\...,"POLYGON ((360356.027 1411250.111, 360235.899 1..."
631402,"POLYGON ((395614.025 1480093.786, 395870.025 1...","[0.20833333333333334, 0.0, 0.0, 0.0, 0.0, 0.0,...","[0.20833333333333334, 0.0, 0.0, -0.33333333333...","[-1.461855507383653, 2.5955461354892018]",0,train,data/MS_NAIP_2023\ortho_1-1_hc_s_ms011_2023_1\...,"POLYGON ((360356.027 1411250.111, 360235.899 1...",65536.0,13,data/MS_NAIP_2023\ortho_1-1_hc_s_ms027_2023_1\...,"POLYGON ((384474.683 1473287.396, 384343.835 1..."
631402,"POLYGON ((395614.025 1480093.786, 395870.025 1...","[0.20833333333333334, 0.0, 0.0, 0.0, 0.0, 0.0,...","[0.20833333333333334, 0.0, 0.0, -0.33333333333...","[-1.461855507383653, 2.5955461354892018]",0,train,data/MS_NAIP_2023\ortho_1-1_hc_s_ms027_2023_1\...,"POLYGON ((384474.683 1473287.396, 384343.835 1...",65536.0,5,data/MS_NAIP_2023\ortho_1-1_hc_s_ms011_2023_1\...,"POLYGON ((360356.027 1411250.111, 360235.899 1..."
631402,"POLYGON ((395614.025 1480093.786, 395870.025 1...","[0.20833333333333334, 0.0, 0.0, 0.0, 0.0, 0.0,...","[0.20833333333333334, 0.0, 0.0, -0.33333333333...","[-1.461855507383653, 2.5955461354892018]",0,train,data/MS_NAIP_2023\ortho_1-1_hc_s_ms027_2023_1\...,"POLYGON ((384474.683 1473287.396, 384343.835 1...",65536.0,13,data/MS_NAIP_2023\ortho_1-1_hc_s_ms027_2023_1\...,"POLYGON ((384474.683 1473287.396, 384343.835 1..."
1145780,"POLYGON ((458590.025 1415069.786, 458846.025 1...","[0.13580246913580246, 0.7160493827160493, 0.0,...","[0.13580246913580246, 8.285714285714286, 0.0, ...","[7.269467655575128, 1.3904592797537614]",7,train,data/MS_NAIP_2023\ortho_1-1_hc_s_ms083_2023_1\...,"POLYGON ((435312.935 1376057.749, 435249.043 1...",65536.0,41,data/MS_NAIP_2023\ortho_1-1_hc_s_ms083_2023_1\...,"POLYGON ((435312.935 1376057.749, 435249.043 1..."
1122883,"POLYGON ((455774.025 1444509.786, 456030.025 1...","[0.28125, 0.171875, 0.0, 0.0, 0.0, 0.0, 0.1875...","[0.28125, 1.9888392857142858, 0.0, -0.33333333...","[0.8910178578788672, 1.0867089845273188]",11,train,data/MS_NAIP_2023\ortho_1-1_hc_s_ms083_2023_1\...,"POLYGON ((435312.935 1376057.749, 435249.043 1...",65536.0,41,data/MS_NAIP_2023\ortho_1-1_hc_s_ms083_2023_1\...,"POLYGON ((435312.935 1376057.749, 435249.043 1..."
1122883,"POLYGON ((455774.025 1444509.786, 456030.025 1...","[0.28125, 0.171875, 0.0, 0.0, 0.0, 0.0, 0.1875...","[0.28125, 1.9888392857142858, 0.0, -0.33333333...","[0.8910178578788672, 1.0867089845273188]",11,train,data/MS_NAIP_2023\ortho_1-1_hc_s_ms083_2023_1\...,"POLYGON ((435312.935 1376057.749, 435249.043 1...",65536.0,67,data/MS_NAIP_2023\ortho_1-1_hc_s_ms135_2023_1\...,"POLYGON ((441474.074 1431469.137, 441410.122 1..."
1122883,"POLYGON ((455774.025 1444509.786, 456030.025 1...","[0.28125, 0.171875, 0.0, 0.0, 0.0, 0.0, 0.1875...","[0.28125, 1.9888392857142858, 0.0, -0.33333333...","[0.8910178578788672, 1.0867089845273188]",11,train,data/MS_NAIP_2023\ortho_1-1_hc_s_ms135_2023_1\...,"POLYGON ((441474.074 1431469.137, 441410.122 1...",65536.0,41,data/MS_NAIP_2023\ortho_1-1_hc_s_ms083_2023_1\...,"POLYGON ((435312.935 1376057.749, 435249.043 1..."
1122883,"POLYGON ((455774.025 1444509.786, 456030.025 1...","[0.28125, 0.171875, 0.0, 0.0, 0.0, 0.0, 0.1875...","[0.28125, 1.9888392857142858, 0.0, -0.33333333...","[0.8910178578788672, 1.0867089845273188]",11,train,data/MS_NAIP_2023\ortho_1-1_hc_s_ms135_2023_1\...,"POLYGON ((441474.074 1431469.137, 441410.122 1...",65536.0,67,data/MS_NAIP_2023\ortho_1-1_hc_s_ms135_2023_1\...,"POLYG

In [9]:
# get footprints of each raster

raster_files = glob('data/MS_NAIP_2023/*/*_1m.tif')
raster_footprints = []
for raster_file in raster_files:
    print(raster_file)
    
    with rio.open(raster_file) as src:
        nodata_mask = src.read_masks(1)
    #     mask = src.read_masks(1)
    #     geoms = list(rio.features.shapes(rio.band(src, 1), mask=mask, transform=src.transform))
    break

data/MS_NAIP_2023\ortho_1-1_hc_s_ms001_2023_1\ortho_1-1_hc_s_ms001_2023_1_1m.tif


<!-- ## Sampling Points for Accuracy Assesment -->

In [67]:
samples['split'].unique()

array(['pretrain', 'pretrain_val', 'train', 'val', 'test'], dtype=object)